# Breast Cancer Wisconsin (Original) - Pre-ML Analysis

**Dataset**: Breast Cancer Wisconsin (Original)

**Descripción**: Dataset de diagnóstico de cáncer de mama obtenido del University of Wisconsin Hospitals. Contiene características de células obtenidas mediante aspiración con aguja fina (FNA) de masas mamarias.

**Objetivo ML**: Clasificación binaria - predecir si un tumor es **benigno (2)** o **maligno (4)** basándose en 9 características celulares.

**Features**: 9 atributos numéricos (escala 1-10): Clump Thickness, Uniformity of Cell Size, Uniformity of Cell Shape, Marginal Adhesion, Single Epithelial Cell Size, Bare Nuclei, Bland Chromatin, Normal Nucleoli, Mitoses.

**Target**: Class (2 = benign, 4 = malignant)

**Particularidades**:
- Incluye ID del paciente (no predictivo, se debe eliminar)
- 16 valores faltantes en columna 'Bare Nuclei' (marcados como '?')
- Clases desbalanceadas: 65.5% benign, 34.5% malignant

---

## PASO 1: Cargar dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Definir nombres de columnas según documentación
column_names = [
    'id',
    'clump_thickness',
    'uniformity_cell_size',
    'uniformity_cell_shape',
    'marginal_adhesion',
    'single_epithelial_cell_size',
    'bare_nuclei',
    'bland_chromatin',
    'normal_nucleoli',
    'mitoses',
    'class'
]

# Cargar dataset (? como NaN)
df = pd.read_csv('breast-cancer-wisconsin.data', 
                 names=column_names,
                 na_values='?')

## PASO 2: Ver estructura

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.dtypes.to_frame(name='dtype')

In [ ]:
df.info()

## PASO 3: Valores faltantes

In [ ]:
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing_Count': missing_counts,
    'Missing_Pct': missing_pct
})

missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

if not missing_df.empty:
    display(missing_df)

## PASO 4: Gráfico simple del target

In [ ]:
# Distribución de clases
class_counts = df['class'].value_counts().sort_index()

plt.figure(figsize=(8, 5))
class_counts.plot(kind='bar', color=['lightgreen', 'salmon'])
plt.title('Distribución de Clases: Benigno (2) vs Maligno (4)', fontsize=14)
plt.xlabel('Clase')
plt.ylabel('Frecuencia')
plt.xticks([0, 1], ['Benigno (2)', 'Maligno (4)'], rotation=0)
plt.grid(axis='y', alpha=0.3)

# Agregar porcentajes
total = len(df)
for i, v in enumerate(class_counts):
    plt.text(i, v + 10, f'{v}\n({v/total*100:.1f}%)', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

---
## LIMPIEZA

### Eliminar columna ID (no predictiva)

In [ ]:
df = df.drop('id', axis=1)

### Tratar valores faltantes

Solo hay 16 valores faltantes (~2.3%) en 'bare_nuclei'. Estrategia: eliminar filas.

In [ ]:
# Eliminar filas con valores faltantes
df_clean = df.dropna()

print(f"Filas antes: {len(df)}")
print(f"Filas después: {len(df_clean)}")
print(f"Filas eliminadas: {len(df) - len(df_clean)}")

### Verificar outliers (opcional - solo inspección visual)

In [ ]:
df_clean.describe()

**Observación**: Todos los features están en escala 1-10 según documentación. No hay outliers evidentes.

---
## PREPROCESAMIENTO

### Encoding

No hay variables categóricas. Solo convertir target a binario (0 = benign, 1 = malignant).

In [ ]:
# Convertir clase: 2 (benign) -> 0, 4 (malignant) -> 1
df_clean['class'] = (df_clean['class'] == 4).astype(int)

df_clean['class'].value_counts().sort_index()

### Separar features (X) y target (y)

In [ ]:
X = df_clean.drop('class', axis=1)
y = df_clean['class']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

### Train/Test Split Manual (80-20)

In [ ]:
# Fijar semilla
np.random.seed(42)

# Crear índices aleatorios
n = len(X)
indices = np.random.permutation(n)

# Split 80-20
split_idx = int(0.8 * n)
train_idx = indices[:split_idx]
test_idx = indices[split_idx:]

# Separar datos
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print(f"Train size: {len(X_train)} ({len(X_train)/n*100:.1f}%)")
print(f"Test size: {len(X_test)} ({len(X_test)/n*100:.1f}%)")

### Convertir a float ANTES del escalado

In [ ]:
X_train = X_train.astype(float)
X_test = X_test.astype(float)

### Feature Scaling Manual (Standardization)

In [ ]:
# Calcular media y std del train
train_mean = X_train.mean()
train_std = X_train.std()

# Aplicar scaling
X_train_scaled = (X_train - train_mean) / train_std
X_test_scaled = (X_test - train_mean) / train_std

# Asegurar tipo float después de scaling
X_train_scaled = X_train_scaled.astype(float)
X_test_scaled = X_test_scaled.astype(float)

print("Escalado completado")
print(f"Train mean (después): {X_train_scaled.mean().mean():.6f}")
print(f"Train std (después): {X_train_scaled.std().mean():.6f}")

---
## VALIDACIÓN

### 1. Verificar Data Leakage

In [ ]:
# Verificar que no hay índices compartidos
train_indices = set(X_train.index)
test_indices = set(X_test.index)
overlap = train_indices.intersection(test_indices)

print(f"Índices en train: {len(train_indices)}")
print(f"Índices en test: {len(test_indices)}")
print(f"Overlap: {len(overlap)}")
print(f"✓ Sin data leakage" if len(overlap) == 0 else "✗ HAY DATA LEAKAGE")

### 2. Verificar NaN e infinitos

In [ ]:
# Verificar NaN
train_nan = X_train_scaled.isnull().sum().sum()
test_nan = X_test_scaled.isnull().sum().sum()

# Verificar infinitos
train_inf = np.isinf(X_train_scaled).sum().sum()
test_inf = np.isinf(X_test_scaled).sum().sum()

print(f"Train - NaN: {train_nan}, Inf: {train_inf}")
print(f"Test - NaN: {test_nan}, Inf: {test_inf}")
print(f"✓ Sin problemas" if (train_nan + test_nan + train_inf + test_inf) == 0 else "✗ HAY PROBLEMAS")

### 3. Comparar distribuciones Train vs Test

In [ ]:
# Distribución del target
train_target_dist = y_train.value_counts(normalize=True).sort_index()
test_target_dist = y_test.value_counts(normalize=True).sort_index()

comparison = pd.DataFrame({
    'Train': train_target_dist,
    'Test': test_target_dist
})

comparison.index = ['Benign (0)', 'Malignant (1)']
comparison

---
## RESUMEN FINAL

In [ ]:
print("="*60)
print("DATASET PREPARADO PARA ML")
print("="*60)
print(f"\nDatos originales: {len(df)} filas")
print(f"Datos limpios: {len(df_clean)} filas")
print(f"Features: {X_train_scaled.shape[1]}")
print(f"\nTrain: {X_train_scaled.shape}")
print(f"Test: {X_test_scaled.shape}")
print(f"\nClases:")
print(f"  - Benign (0): {(y_train == 0).sum()} train, {(y_test == 0).sum()} test")
print(f"  - Malignant (1): {(y_train == 1).sum()} train, {(y_test == 1).sum()} test")
print("\n" + "="*60)